# Exploratory Data Analysis: California Wildfires, 2016–2020

Run `data_cleaning.ipynb` first. Every figure is saved to `figures/` for the report and slides.
Under each plot, **What to look for** says what to check and note down.

In [ ]:
import sys; sys.path.append("..")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data import load, feature_sets, cv_folds, WEATHER

df = load()
sets = feature_sets(df)
order = sorted(df["TARGET"].unique())
df["LARGE"] = df["TARGET"].isin(["D-E", "F-G"])     # 100+ acres (class D and up)

FIG = Path("../figures"); FIG.mkdir(exist_ok=True)
def save(name):
    plt.savefig(FIG / f"{name}.png", dpi=200, bbox_inches="tight")

print(df.shape)
print(df["FIRE_YEAR"].value_counts().sort_index())

## 1. How imbalanced is the target?

In [ ]:
counts = df["TARGET"].value_counts().reindex(order)
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(order, counts, color="firebrick")
ax.set_yscale("log")
ax.set(title="California fires per size class, 2016–2020 (log scale)", xlabel="Size class", ylabel="Number of fires")
for b, v in zip(bars, counts):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,}\n({v / len(df):.1%})", ha="center", va="bottom", fontsize=8)
save("01_class_imbalance"); plt.show()

**What to look for:** the share of class A. A model that always guesses A scores that much accuracy, which is why you use macro-F1 and balanced accuracy.

## 2. Where do fires happen, and where are the large ones?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
small = df[~df["LARGE"]]
ax.scatter(small["LONGITUDE"], small["LATITUDE"], s=1, c="lightgray", label="A–C (under 100 acres)")
colors = {"D-E": "orange", "F-G": "darkred"}
for cls in ["D-E", "F-G"]:
    s = df[df["TARGET"] == cls]
    ax.scatter(s["LONGITUDE"], s["LATITUDE"], s=8, c=colors[cls], label=cls)
ax.set(title="California fire locations", xlabel="Longitude", ylabel="Latitude")
ax.set_aspect(1.2)
ax.legend(markerscale=3, loc="upper right")
save("02_fire_map"); plt.show()

**What to look for:** are large fires clustered (e.g., Southern California, the Sierra foothills, the north) or spread out? Clusters mean the location split matters a lot.

## 3. The location split: which regions land in which fold?

In [ ]:
df["FOLD"] = -1
for i, (tr, te) in enumerate(cv_folds(df)):
    df.loc[te.index, "FOLD"] = i

fig, ax = plt.subplots(figsize=(7, 8))
for i in sorted(df["FOLD"].unique()):
    s = df[df["FOLD"] == i]
    ax.scatter(s["LONGITUDE"], s["LATITUDE"], s=1, label=f"fold {i}")
ax.set(title="Cross-validation folds (each color = regions held out together)", xlabel="Longitude", ylabel="Latitude")
ax.set_aspect(1.2)
ax.legend(markerscale=10)
save("03_cv_folds_map"); plt.show()

pd.crosstab(df["TARGET"], df["FOLD"], margins=True)

**What to look for:** the map should look like a patchwork of 0.5° squares. In the table, every fold should have a reasonable number of F-G fires. If one fold has very few, the folds need rebalancing before the final results.

## 4. Do conditions differ between small and large fires?

In [ ]:
feats = ["erc", "vpd", "fm1000", "fm100", "vs", "tmmx", "rmin", "NDVI-1day", "CheatGrass", "Elevation", "Slope", "erc_Percentile"]
feats = [f for f in feats if f in df.columns]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, f in zip(axes.flat, feats):
    data = [df.loc[df["TARGET"] == c, f].dropna() for c in order]
    ax.boxplot(data, tick_labels=order, showfliers=False)
    ax.set_title(f)
for ax in axes.flat[len(feats):]:
    ax.axis("off")
plt.suptitle("Feature values by fire size class (outliers hidden)")
plt.tight_layout()
save("04_features_by_class"); plt.show()

# The same idea as numbers: median of each feature per class
df.groupby("TARGET")[feats].median().round(2)

**What to look for:** features whose boxes shift steadily from A to F-G are likely strong predictors.
Expected: higher `erc`/`vpd`/`tmmx`, lower `fm100`/`fm1000`/`rmin` (drier fuels) for larger fires.
Write down your guesses now and compare them with the SHAP results later.

## 5. Seasonality: when do large fires happen?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df["DISCOVERY_MONTH"].value_counts().sort_index().plot.bar(ax=axes[0], color="gray")
axes[0].set(title="Number of fires by month", xlabel="Month", ylabel="Fires")

(df.groupby("DISCOVERY_MONTH")["LARGE"].mean() * 100).plot.bar(ax=axes[1], color="firebrick")
axes[1].set(title="Share of fires that became large (100+ acres)", xlabel="Month", ylabel="% large")

plt.tight_layout()
save("05_seasonality"); plt.show()

**What to look for:** whether the months with the *most* fires are also the months with the highest *share* of large fires. They often aren't the same.

## 6. Vegetation: which land cover types produce large fires?

In [ ]:
lc_cols = [c for c in df.columns if c.startswith("LC_")]
lc = pd.DataFrame({
    "fires": [int(df[c].sum()) for c in lc_cols],
    "% large": [df.loc[df[c] == 1, "LARGE"].mean() * 100 for c in lc_cols],
}, index=lc_cols)
lc = lc[lc["fires"] >= 100].sort_values("% large")   # skip tiny categories

fig, ax = plt.subplots(figsize=(7, 0.35 * len(lc) + 1))
ax.barh(lc.index, lc["% large"], color="olivedrab")
ax.axvline(df["LARGE"].mean() * 100, color="k", ls="--", label="overall average")
ax.set(title="Share of fires that became large, by land cover", xlabel="% large (100+ acres)")
ax.legend()
save("06_landcover_large_share"); plt.show()
lc.round(1)

**What to look for:** land cover types well above the dashed line. Look up the codes in the LANDFIRE / NLCD tables (e.g., 52 = shrub/scrub, 71 = grassland, 42 = evergreen forest in NLCD).

## 7. Slope direction: do south-facing slopes produce more large fires?

In [ ]:
if "Aspect_cos" in df.columns:
    facing = pd.cut(df["Aspect_cos"], bins=[-1.01, -0.5, 0.5, 1.01],
                    labels=["south-facing", "east/west-facing", "north-facing"])
    share = df.groupby(facing, observed=True)["LARGE"].mean() * 100
    ax = share.plot.bar(color="sienna", rot=0, figsize=(6, 4))
    ax.set(title="Share of large fires by slope direction", ylabel="% large", xlabel="")
    save("07_aspect"); plt.show()
    print(share.round(2))

**What to look for:** if south-facing slopes show a higher share, that supports the sin/cos encoding and gives a hypothesis to check with SHAP.

## 8. Which weather features are redundant?

In [ ]:
wx = [c for c in ["pr", "tmmn", "tmmx", "rmin", "rmax", "sph", "vs", "srad", "etr", "vpd", "fm100", "fm1000", "bi", "erc"] if c in df.columns]
corr = df[wx].corr()

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(wx)), wx, rotation=45)
ax.set_yticks(range(len(wx)), wx)
for i in range(len(wx)):
    for j in range(len(wx)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
plt.colorbar(im)
ax.set_title("Correlation between day-of-ignition weather features")
save("08_weather_correlation"); plt.show()

# Pairs that carry nearly the same information
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
pairs[pairs.abs() > 0.85].sort_values(ascending=False)

**What to look for:** pairs above about 0.85 carry almost the same information. SHAP may split credit between them, so mention this when interpreting feature importance.

## 9. How much is missing in the model features?

In [ ]:
miss = df[sets["model3_all"]].isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]
if len(miss):
    ax = (miss * 100).plot.barh(figsize=(7, 0.3 * len(miss) + 1), color="gray")
    ax.set(title="Missing values in model features (filled with the training median)", xlabel="% missing")
    ax.invert_yaxis()
    save("09_missing"); plt.show()
miss.round(3)

**What to look for:** any feature with a lot missing (say 20%+). Median-filling a feature that is mostly empty weakens it; mention it as a limitation.

## Notes for the report

Fill in after running:
- Class balance: …
- Where large fires cluster: …
- Strongest-looking features from section 4: …
- Peak month for large fires: …
- Land cover types with the highest large-fire share: …
- South- vs north-facing slopes: …
- Redundant weather pairs: …